In [9]:
from google import genai
from google.genai import types
import pandas as pd
import google.generativeai as genai`
import json
import os.path

import gspread
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from datetime import datetime, timedelta

def main():
    today = datetime.today()
    first_day_this_month = today.replace(day=1)

    # Subtract one day to get the last day of the previous month
    last_day_prev_month = first_day_this_month - timedelta(days=1)
    
    df = extract_financials_with_gemini(r"C:\Users\mochamad.ilmawan\OneDrive - semenindonesia.com\Renstra\Dashboard KPI Financial Aspect\Dashboard KPI Renstra_send.xlsx")
    print(df)
    # Convert month name to a 2026 datetime object, then shift to Month End
    if df is not None and not df.empty:
        print("Data extracted successfully.")
        
        # 1. Convert month name to last day of month
        #if 'Month' in df.columns:
        #    df['Month'] = pd.to_datetime(df['Month'] + ' 2026', format='%B %Y') + pd.offsets.MonthEnd(0)
        #    df['Month'] = df['Month'].dt.strftime('%d/%m/%Y')

        # 2. IMPORTANT: Remove the line 'df_flattened = pd.DataFrame(rows)' 
        # Your 'df' is already the flattened dataframe from Gemini!

        # 3. Append to Sheets
        # Note: Make sure connect_to_sheet() is working
        creds = connect_to_sheet()
        
        # Append Raw Data
        append_sheet('1taze3KNq8g1U76oSjp1yN_jMu6lzCZJiin0BQX5LzTs', 'COGS Variance V!A:Z', None, creds, df)
        
        # 4. Pivot Data
        # df_horizon = get_rkap(df)
        
        # Append Pivoted Data
        # append_sheet('1G1UbCcaPjnkB-f-vUY3gQ3vgw03Lj4kLlNIQjYRITKY', 'Sheet2!A:Z', None, creds, df_horizon)
        
        #print(df_horizon)
    else:
        print("Failed to extract data. Check Gemini response or JSON format.")
    
def get_rkap(df):
    df.columns = ['No','Account','Month','RKAP','Real']
    
    df_rkap = df[['Account', 'RKAP', 'Month']].copy()
    df_rkap.rename(columns={'RKAP': 'Value'}, inplace=True)
    df_rkap['Account'] = df_rkap['Account'] + ' RKAP'
    
    df_real = df[['Account', 'Real', 'Month']].copy()
    df_real.rename(columns={'Real': 'Value'}, inplace=True)
    df_real['Account'] = df_real['Account'] + ' Real'
    
    df = pd.concat([ df_rkap, df_real], axis=0).reset_index(drop=True)
    
    #df['date'] = '31/03/2026'
    
    df_pivoted = df.pivot_table(
        index=['Month'], 
        columns='Account', 
        values='Value',
        aggfunc='first' # Use 'first' to keep the value as is
    ).reset_index()
    
    return df_pivoted
    
def connect_to_sheet():
    SCOPES = ['https://www.googleapis.com/auth/spreadsheets']
    creds = None
    # The file token.json stores the user's access and refresh tokens, and is
    # created automatically when the authorization flow completes for the first
    # time.
    if os.path.exists(r'C:\Users\mochamad.ilmawan\token.json'):
        creds = Credentials.from_authorized_user_file(r'C:\Users\mochamad.ilmawan\token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'D:\OneDrive\Documents\Direct Selling\MONITORING\Monitoring 2023\client_secret_738271294618-6g8e0hle4jpq8nau1d7b3q9jfsgp0ruk.apps.googleusercontent.com.json', SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open(r'C:\Users\mochamad.ilmawan\token.json', 'w') as token:
            token.write(creds.to_json())
    return creds

def get_excel_data(address, sheet_name=0, usecols=None, skiprows=0, nrows=None, header=None):
    df = pd.read_excel(
        address,
        sheet_name=sheet_name,
        usecols=usecols,
        skiprows=skiprows,
        nrows=nrows,
        dtype=str,
        thousands='.', 
        decimal=','
    )
    return df

def append_sheet(spreadSheetId, spread_range, scopes, creds, df):
    df_clean = df.fillna('')
    header = df.columns.tolist()
    data = df.values.tolist()
    
    # Combine them: [header] creates a list within a list to match the Sheets API format
    values_with_header = [header] + data
    try:
        values = values_with_header
        service = build('sheets', 'v4', credentials=creds)
        body = {
            'values': values_with_header
        }
        result = service.spreadsheets().values().append(
            spreadsheetId=spreadSheetId, 
            range=spread_range, 
            valueInputOption="USER_ENTERED", 
            body=body).execute()
        print(f"{(result.get('updates').get('updatedCells'))} cells appended.")
        return result

    except HttpError as error:
        print(f"An error occurred: {error}")
        return error
    
def extract_financials_with_gemini(file_path):
    # 1. Setup Gemini API
    # Replace 'YOUR_API_KEY' with your actual Google AI Studio API key
    genai.configure(api_key="AQ.Ab8RN6ItXczDB3cRfHa4eS8b1pkKd4DWyi0CKyly65pCCMPsYw")
    generation_config = {
        "temperature": 0,
        "top_p": 0.95,
        "response_mime_type": "application/json",
    }
    model = genai.GenerativeModel(
      model_name="gemini-2.5-flash", # or 2.0
      generation_config=generation_config,
    )
    # 2. Load the Excel data
    # Reading as string helps if the Excel is messy or has multiple headers
    df_raw = get_excel_data(file_path, 'AVAR COGS', "B:U", 1, 60, 1)
    raw_data_string = df_raw.to_string()

    # 3. Construct the prompt
    prompt = f"""
    Act as a precise data extraction engine. Extract financial metrics from the provided raw Excel dump.

    ### EXTRACTION RULES:
    1. MAPPING: Locate the exact row for each of the 19 metrics listed below.
    2. CONSISTENCY: Based on data you also have to generate group column (Bahan Baku, Bahan Bakar, Listrik, Kemasan, Perniagaan, and Other).
    3. VALUES: 
       - Extract 'Indeks RKAP', 'Indeks Real', 'Indeks Var', 'Volume RKAP', 'Volume Real', 'Volume Var', etc values as numbers.
       - If a value is missing, null, or "-" in the raw data, return 0.0. 
       - DO NOT use nested dictionaries. Use a flat structure.
       - Fill Month Value with today
    4. FORMAT: Return ONLY a valid JSON list of objects. No conversational text.

    ### METRIC LIST:
    1. Batu kapur
    2. Tanah Liat
    3. Pasir Silika
    4. Pasir Besi & Copper Slag
    5. Trass Pozolan
    6. Gypsum
    7. Fly Ash
    8. CGA
    9. Others & TLCC
    10. Batubara
    11. Solar
    12. Sekam Padi
    13. Others & TLCC
    14. Upto Kiln 
    15. FM 
    16. Others 
    17. TLCC
    18. 1 Ply 40 KG
    19. 1 Ply 50 KG
    20. 2 Ply 40 KG
    21. 2 Ply 50 KG
    22. 3 Ply 40 KG
    23. 3 Ply 50 KG
    24. Jumbo 1 ton
    25. Jumbo 2 ton
    26. Polysling
    27. TLCC
    28. STO Laut
    29. STO Darat
    30. PBM
    31. Packer
    31. Lain-lain (TLCC & Elim)
    32. Domestik
    33. Penjualan Semen Bag
    ### REQUIRED JSON STRUCTURE EXAMPLE:
    [
      {{
        "No": 1, 
        "Account": "Batu kapur", 
        "Group": "Bahan Baku", 
        "Month": "19/05/2026", 
        "Indeks RKAP": 1628 , 
        "Indeks Real": 1626 ,
        "Indeks Var": 0.001,
        "Volume (ton/liter/kwh) RKAP": 13000000,
        "Volume (ton/liter/kwh) Real": 13100000,
        "Volume (ton/liter/kwh) Var":-172345,
        "Harga (Rp) RKAP": 162800 , 
        "Harga (Rp) Real": 162600 ,
        "Harga (Rp) Var": -200,
        "Nilai (Rp Miliar) RKAP": 300,
        "Nilai (Rp Miliar) Real": 310,
        "Nilai (Rp Miliar) Var":-10,
        "Dampak (Rp Miliar) Harga": -9,
        "Dampak (Rp Miliar) Indeks": -7,
        "Dampak (Rp Miliar) Volume":-1
      }},
      {{
        "No": 1, 
        "Account": "Tanah Liat", 
        "Group": "Bahan Baku", 
        "Month": "19/05/2026", 
        "Indeks RKAP": 1628 , 
        "Indeks Real": 1626 ,
        "Indeks Var": 0.001,
        "Volume (ton/liter/kwh) RKAP": 13000000,
        "Volume (ton/liter/kwh) Real": 13100000,
        "Volume (ton/liter/kwh) Var":-172345,
        "Harga (Rp) RKAP": 162800 , 
        "Harga (Rp) Real": 162600 ,
        "Harga (Rp) Var": -200,
        "Nilai (Rp Miliar) RKAP": 300,
        "Nilai (Rp Miliar) Real": 310,
        "Nilai (Rp Miliar) Var":-10,
        "Dampak (Rp Miliar) Harga": -9,
        "Dampak (Rp Miliar) Indeks": -7,
        "Dampak (Rp Miliar) Volume":-1
      }}
    ]

    Raw Data:
    {raw_data_string}
    """

    # 4. Generate response
    response = model.generate_content(prompt)
    json_text = response.text.replace('```json', '').replace('```', '').strip()

    print(f"DEBUG: Response Length: {len(json_text)}") # See if it's hitting a limit
    try:
        data = json.loads(json_text)
        # 5. Convert back to DataFrame
        return pd.DataFrame(data)
    except Exception as e:
        print(f"Error parsing JSON: {e}")
        return None

if __name__ == '__main__':
    main()

DEBUG: Response Length: 25844
    No                   Account        Group       Month  Indeks RKAP  \
0    1                Batu kapur   Bahan Baku  19/05/2024     1.627664   
1    2                Tanah Liat   Bahan Baku  19/05/2024     0.268150   
2    3              Pasir Silika   Bahan Baku  19/05/2024     0.050146   
3    4  Pasir Besi & Copper Slag   Bahan Baku  19/05/2024     0.024165   
4    5             Trass Pozolan   Bahan Baku  19/05/2024     0.057426   
5    6                    Gypsum   Bahan Baku  19/05/2024     0.037546   
6    7                   Fly Ash   Bahan Baku  19/05/2024     0.034258   
7    8                       CGA   Bahan Baku  19/05/2024     0.517432   
8    9             Others & TLCC   Bahan Baku  19/05/2024     0.000000   
9   10                  Batubara  Bahan Bakar  19/05/2024     0.210780   
10  11                     Solar  Bahan Bakar  19/05/2024     0.440895   
11  12                Sekam Padi  Bahan Bakar  19/05/2024     0.015737   
12  13  

665 cells appended.
